<a href="https://colab.research.google.com/github/yasuhisasugiura-crypto/newfile_generator/blob/main/lola_train_%EF%BD%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")
!pip -q install -U peft torchao
import torch, transformers, peft, torchao
print(torch.__version__, transformers.__version__, peft.__version__, torchao.__version__, torch.cuda.get_device_name(0))

Mounted at /content/drive


W1121 06:27:11.256000 582 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1121 06:27:11.298000 582 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


2.14.1+cu130 5.22.1 0.22.0 0.18.0 Tesla T4


In [2]:
def answer_prompt(question, contexts):
    context_text = "\n\n".join("[chunk_id=%d] %s\n%s" % (c["chunk_id"], c["title"], c["text"]) for c in contexts)
    return (
        "質問: " + question + "\n\n"
        "次の資料を根拠に、上の質問に短く答えてください。\n"
        "答え方のきまり:\n"
        "- answer には、質問の答えになる語句だけを書く（人名・地名・作品名・数値・日付など）。\n"
        "- 文にしない。「〜です」「〜である」を付けない。説明・補足・括弧書き・読み仮名・英語表記を付けない。\n"
        "- 「どちら」「どっち」と聞かれたら、選んだほうの名前だけを書く。\n"
        "- 「はい」か「いいえ」で答える質問には、「はい」か「いいえ」だけを書く。\n"
        "- 資料のどこにも答えが書かれていない場合だけ、answer を「資料からは特定できません」、chunk_id を null にする。\n"
        "chunk_id には、答えが書かれている資料の chunk_id(整数)を1つだけ書く。\n"
        '形式: {"answer":"答えの語句","chunk_id":整数}\n\n'
        "資料:\n" + context_text + "\n\n質問（再掲）: " + question
    )

import hashlib
md5 = hashlib.md5(answer_prompt("Q", [{"chunk_id": 1, "title": "T", "text": "X"}]).encode("utf-8")).hexdigest()
print(md5, "一致" if md5 == "1e990717ef9ebb51535b233deec1503e" else "【不一致：止める】")

1e990717ef9ebb51535b233deec1503e 一致


In [3]:
RUN_NAME ="d_s3"      # v2_s1 / v2_s2 / v2_t0
SEED = 3               # 1 / 2 / 0
YN_CHUNK ="first"      # "first"（今と同じ） / "title"（質問に出てくる記事名の資料）
SHUFFLE_P = 1.0         # はい/いいえ以外の問題は、すべて資料の並びを入れ替える
ROOT = "/content/drive/MyDrive/LLM講座最終コンペ_lora"
CONTEXTS = ROOT + "/train_contexts_1500.jsonl"
ANSWERS = ROOT + "/train_answers.json"
OUT = ROOT + "/train_v2/" + RUN_NAME
MODEL = "Qwen/Qwen3.5-0.8B"
REVISION = "2fc06364715b967f1860aea9cf38778875588b17"
CTX_TOP_K = 5
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LR, EPOCHS, GRAD_ACCUM, MAX_LEN = 1e-4, 1, 8, 3072
import os
assert not os.path.exists(OUT + "/merged"), "同じ名前の結果がある。RUN_NAME を変える（上書き防止）"

In [4]:
import json, re, unicodedata, random
YESNO = {"yes": "はい", "no": "いいえ"}

def norm(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", s or "").lower())

def yn_chunk(question, contexts):
    if YN_CHUNK == "title":
        q = norm(question)
        for c in contexts:
            t = norm(c["title"])
            if t and t in q:
                return c["chunk_id"]
    return contexts[0]["chunk_id"]

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
gold = json.load(open(ANSWERS, encoding="utf-8"))
items = [json.loads(l) for l in open(CONTEXTS, encoding="utf-8") if l.strip()]
examples, skipped, yn_title = [], {"資料に正解なし": 0, "長すぎる": 0}, 0
shuf_rng = random.Random(SEED + 1000)      # 並べ替え専用の乱数（学習の乱数とは別）
n_shuf, pos_count = 0, [0] * CTX_TOP_K     # 入れ替えた問題の数、教師の資料の位置の分布
for it in items:
    contexts = it["contexts"][:CTX_TOP_K]
    answers = gold[it["q_id"]]["answers"]
    target = None
    if all(norm(a) in YESNO for a in answers):
        cid = yn_chunk(it["question"], contexts)
        yn_title += cid != contexts[0]["chunk_id"]
        target = {"answer": YESNO[norm(answers[0])], "chunk_id": cid}
    else:
        if shuf_rng.random() < SHUFFLE_P:
            contexts = shuf_rng.sample(contexts, len(contexts))
            n_shuf += 1
        for k, c in enumerate(contexts):
            body = norm(c["title"] + c["text"])
            hit = next((a for a in answers if norm(a) and norm(a) in body), None)
            if hit:
                target = {"answer": hit, "chunk_id": c["chunk_id"]}
                pos_count[k] += 1
                break
    if target is None:
        skipped["資料に正解なし"] += 1
        continue
    prompt = tok.apply_chat_template([{"role": "user", "content": answer_prompt(it["question"], contexts)}],
                                     tokenize=False, add_generation_prompt=True, enable_thinking=False)
    p_ids = tok(prompt, add_special_tokens=False).input_ids
    t_ids = tok(json.dumps(target, ensure_ascii=False, separators=(",", ":")) + "<|im_end|",
                add_special_tokens=False).input_ids
    if len(p_ids) + len(t_ids) > MAX_LEN:
        skipped["長すぎる"] += 1
        continue
    examples.append((p_ids, t_ids))

print("質問 %d / 学習に使う %d / 除外 %s / はい・いいえで1件目以外を根拠にした %d 問" % (len(items), len(examples), skipped, yn_title))
print("資料の並びを入れ替えた問題: %d / 教師の資料の位置（はい・いいえ以外）: %s" % (n_shuf, pos_count))
assert 1030 <= len(examples) <= 1045, "学習に使う数が 1039 前後ではないので止める"
assert pos_count[0] < 0.70 * sum(pos_count), "1番目の割合が下がっていないので止める"


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

質問 1500 / 学習に使う 1038 / 除外 {'資料に正解なし': 461, '長すぎる': 1} / はい・いいえで1件目以外を根拠にした 0 問
資料の並びを入れ替えた問題: 1210 / 教師の資料の位置（はい・いいえ以外）: [456, 137, 69, 54, 33]


In [5]:
import math, time, sys, random
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REVISION, dtype=torch.bfloat16).to("cuda")
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.config.use_cache = False
model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                         target_modules="all-linear", task_type="CAUSAL_LM"))
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.0)
steps = math.ceil(len(examples) * EPOCHS / GRAD_ACCUM)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: min(1.0, (s + 1) / max(1, steps // 10)) * max(0.0, 1 - s / steps))
model.train()
t0, step, run_loss, n_loss = time.time(), 0, 0.0, 0
for ep in range(EPOCHS):
    random.shuffle(examples)
    for i, (p_ids, t_ids) in enumerate(examples, 1):
        ids = torch.tensor([p_ids + t_ids], device="cuda")
        labels = torch.tensor([[-100] * len(p_ids) + t_ids], device="cuda")
        loss = model(input_ids=ids, labels=labels).loss / GRAD_ACCUM
        loss.backward()
        run_loss += loss.item() * GRAD_ACCUM; n_loss += 1
        if i % GRAD_ACCUM == 0 or i == len(examples):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
            step += 1
            if step % 10 == 0 or step == steps:
                print("step %d/%d loss %.4f（%.0f 分）" % (step, steps, run_loss / max(1, n_loss), (time.time() - t0) / 60), flush=True)
                run_loss, n_loss = 0.0, 0

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


step 10/130 loss 0.9797（8 分）
step 20/130 loss 0.1539（16 分）
step 30/130 loss 0.1344（24 分）
step 40/130 loss 0.1021（32 分）
step 50/130 loss 0.1130（40 分）
step 60/130 loss 0.1280（48 分）
step 70/130 loss 0.1068（55 分）
step 80/130 loss 0.1267（63 分）
step 90/130 loss 0.1357（71 分）
step 100/130 loss 0.1063（79 分）
step 110/130 loss 0.0898（87 分）
step 120/130 loss 0.0758（95 分）
step 130/130 loss 0.0905（103 分）


In [6]:
os.makedirs(OUT, exist_ok=True)
model.save_pretrained(OUT + "/adapter")
merged = model.merge_and_unload().to(torch.float16)
merged.save_pretrained(OUT + "/merged")
tok.save_pretrained(OUT + "/merged")
json.dump({"run": RUN_NAME, "seed": SEED, "yn_chunk": YN_CHUNK, "shuffle_p": SHUFFLE_P, "examples": len(examples), "steps": steps,
           "minutes": round((time.time() - t0) / 60, 1), "prompt_md5": md5},
          open(OUT + "/train_info.json", "w"), ensure_ascii=False, indent=1)
print("保存:", OUT + "/merged")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

保存: /content/drive/MyDrive/LLM講座最終コンペ_lora/train_v2/d_s3/merged
